# Antibiotic Resistance Evolution Simulator
### INT6064 Individual Teaching Module — two-lesson secondary STEM module

**What this is:** a small, runnable model of how **antibiotic resistance** evolves by natural selection in a population of bacteria.
Students change the environment and the parameters, watch the frequency of resistant bacteria over generations, and explain the outcomes — building **computational thinking** (decomposition, abstraction, algorithms, pattern recognition, evaluation, debugging) inside a real **biology + mathematics** context.

**How to use:** run the cells top to bottom in Google Colab or Jupyter. Then use the "student experiment" cell to test **one variable at a time**. 

## 1. How the model works

A population of `N` bacteria carries one of two genotypes:
- `S` = susceptible to the antibiotic
- `R` = resistant

Each generation repeats the same loop:
1. **Selection** — bacteria reproduce in proportion to their *fitness* in the current environment (antibiotic ON or OFF).
2. **Reproduction** — `N` parents are sampled (weighted by fitness); each child inherits the parent's genotype.
3. **Record** — log the frequency of `R` for that generation.

The whole model is just that loop repeated many times. That loop is the **algorithm**. 

In [ ]:
import random

class Fitness:
    """Relative reproductive success (teaching values, not clinical data)."""
    def __init__(self, s_no_abx=1.00, r_no_abx=0.95, s_on_abx=0.15, r_on_abx=0.95):
        self.s_no_abx, self.r_no_abx = s_no_abx, r_no_abx
        self.s_on_abx, self.r_on_abx = s_on_abx, r_on_abx
    def weights(self, antibiotic_on):
        return (self.s_on_abx, self.r_on_abx) if antibiotic_on else (self.s_no_abx, self.r_no_abx)

def run_simulation(n=1000, generations=40, f0_resistant=0.01,
                   antibiotic_schedule=lambda g: True, fitness=None, mutation_rate=0.0, seed=1):
    """Return the resistant frequency at each generation."""
    rng = random.Random(seed)
    if fitness is None: fitness = Fitness()
    n_res = round(n * f0_resistant)
    population = ["R"] * n_res + ["S"] * (n - n_res)
    history = []
    for gen in range(generations):
        wS, wR = fitness.weights(antibiotic_schedule(gen))
        weights = [wR if gt == "R" else wS for gt in population]
        history.append(population.count("R") / len(population))
        new_pop = rng.choices(population, weights=weights, k=len(population))
        if mutation_rate > 0:
            for i, gt in enumerate(new_pop):
                if rng.random() < mutation_rate:
                    new_pop[i] = "S" if gt == "R" else "R"
        population = new_pop
    history.append(population.count("R") / len(population))
    return history

print("Model loaded.")

## 2. First run — predict, then check (Lesson 1)

**Before you run:** if we start with only **1% resistant** bacteria and add the antibiotic, what do you think happens to the resistant % over 40 generations? Write your prediction, then run. 

In [ ]:
import matplotlib.pyplot as plt

hist = run_simulation(n=1000, generations=40, f0_resistant=0.01,
                      antibiotic_schedule=lambda g: True, seed=1)
plt.figure(figsize=(7,4))
plt.plot([100*x for x in hist], marker="o", ms=3)
plt.xlabel("Generation"); plt.ylabel("Resistant bacteria (%)")
plt.title("Full antibiotic course: resistance rises")
plt.ylim(-5,105); plt.grid(alpha=.3); plt.show()
print("final resistant %:", round(hist[-1]*100,1))

### What did you see?
- Where does the curve rise fastest? Why?
- Connect it to **natural selection**: which bacteria survive and reproduce when the antibiotic is present? 

## 3. Compare environments (the "cost" of resistance)

Now run with **no antibiotic**. Resistance often carries a small survival cost, so the resistant % can *fall* when the drug is removed. 

In [ ]:
no_abx = run_simulation(n=1000, generations=40, f0_resistant=0.50,
                        antibiotic_schedule=lambda g: False, seed=1)
sub_lethal = run_simulation(n=1000, generations=40, f0_resistant=0.01,
                            antibiotic_schedule=lambda g: True,
                            fitness=Fitness(s_on_abx=0.60, r_on_abx=0.95), seed=1)
small = run_simulation(n=50, generations=40, f0_resistant=0.30,
                       antibiotic_schedule=lambda g: False, seed=3)

plt.figure(figsize=(7,4))
plt.plot([100*x for x in no_abx], label="No antibiotic (cost of resistance)")
plt.plot([100*x for x in sub_lethal], label="Sub-lethal / half dose")
plt.plot([100*x for x in small], label="Small population (drift)")
plt.xlabel("Generation"); plt.ylabel("Resistant (%)"); plt.legend(); plt.grid(alpha=.3); plt.show()

## 4. Student experiment (Lesson 2) — change ONE variable

Pick **one** thing to change and keep everything else fixed (fair test):
- initial resistant frequency `f0_resistant`
- dose / selection strength (e.g. `Fitness(s_on_abx=...)`)
- population size `n` (drift)
- number of generations

Write your prediction first, edit the cell, run, and compare the curve. 

In [ ]:
# === EDIT ONLY THESE LINES ===
n = 1000              # try 50, 200, 1000 ...
gens = 40
f0 = 0.05             # try 0.01, 0.05, 0.30 ...
dose_s = 0.15         # susceptible fitness under drug: 0.15 (full) ... 0.60 (half) ... 1.0 (none)
# =============================

result = run_simulation(n=n, generations=gens, f0_resistant=f0,
                        antibiotic_schedule=lambda g: True,
                        fitness=Fitness(s_on_abx=dose_s, r_on_abx=0.95), seed=1)
plt.figure(figsize=(7,4)); plt.plot([100*x for x in result])
plt.xlabel("Generation"); plt.ylabel("Resistant (%)")
plt.title(f"N={n}, f0={f0}, susceptible-fitness={dose_s}"); plt.grid(alpha=.3); plt.show()
print("final resistant %:", round(result[-1]*100,1))

## 5. Explain and decide (CER)

For your experiment, write:
- **Claim:** what happened to the resistant %?
- **Evidence:** describe the curve / numbers (data).
- **Reasoning:** why did it happen (selection, fitness, drift, cost)?

**Big question:** Based on your evidence, what advice would you give about how people should use antibiotics? 